# 7. Cherry-pick

`git cherry-pick` copies **one commit** onto the branch you are on.

When is that useful? The barge software was delivered to a customer from
a `release` branch. Then a bug was found and fixed on `main` — but `main`
also has new, untested features. The customer needs the fix, and only the
fix.

```mermaid
gitGraph
    commit id: "volume + froude (bug)"
    branch release
    checkout main
    commit id: "fix"
    commit id: "tonnes per cm"
    commit id: "wetted surface"
    checkout release
    cherry-pick id: "fix"
```

In [1]:
import os, shutil, stat, sys
from pathlib import Path


def remove_readonly(func, path, _):
    os.chmod(path, stat.S_IWRITE)  # git makes its object files read-only
    func(path)


work = Path.home() / "git-course" / "07-cherry-pick"
if work.exists():
    shutil.rmtree(work, onexc=remove_readonly)
work.mkdir(parents=True)
%cd {work}

/home/emircem/git-course/07-cherry-pick


## Starting point

The delivered code. The Froude number is missing a square root:

In [2]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def froude_number(speed, length):
    """Speed-length ratio that scales wave resistance."""
    return speed / (9.81 * length)

Writing hydrostatics.py


In [3]:
%%writefile test_hydrostatics.py
from hydrostatics import froude_number

# An 80 m hull at 12 knots (6.17 m/s) should have Fn of about 0.22.
assert 0.21 < froude_number(6.17, 80.0) < 0.23

print("all checks passed")

Writing test_hydrostatics.py


In [4]:
!git init
!git add hydrostatics.py test_hydrostatics.py
!git commit -m "Add volume and the Froude number"

Initialized empty Git repository in /home/emircem/git-course/07-cherry-pick/.git/
[main (root-commit) 4d94b4f] Add volume and the Froude number
 2 files changed, 19 insertions(+)
 create mode 100644 hydrostatics.py
 create mode 100644 test_hydrostatics.py


This is the delivered version, so we put a `release` branch here:

In [5]:
!git branch release

## Work continues on main

First, the fix:

In [6]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def froude_number(speed, length):
    """Speed-length ratio that scales wave resistance."""
    return speed / (9.81 * length) ** 0.5

Overwriting hydrostatics.py


In [7]:
!git add hydrostatics.py
!git commit -m "Fix the Froude number: take the square root"

[main 91efc42] Fix the Froude number: take the square root
 1 file changed, 1 insertion(+), 1 deletion(-)


Then two new features:

In [8]:
%%writefile -a hydrostatics.py


def tonnes_per_cm(length, beam):
    """Tonnes of cargo that sink the barge by one centimetre."""
    return length * beam * RHO / 100000.0

Appending to hydrostatics.py


In [9]:
!git add hydrostatics.py
!git commit -m "Add tonnes per centimetre immersion"

[main 257f990] Add tonnes per centimetre immersion
 1 file changed, 5 insertions(+)


In [10]:
%%writefile -a hydrostatics.py


def wetted_surface(length, beam, draft):
    """Rough wetted surface area, in m^2."""
    return length * (beam + 2 * draft)

Appending to hydrostatics.py


In [11]:
!git add hydrostatics.py
!git commit -m "Add a rough wetted surface estimate"

[main 539c065] Add a rough wetted surface estimate
 1 file changed, 5 insertions(+)


In [12]:
!git log --oneline --graph --all

* 539c065 (HEAD -> main) Add a rough wetted surface estimate
* 257f990 Add tonnes per centimetre immersion
* 91efc42 Fix the Froude number: take the square root
* 4d94b4f (release) Add volume and the Froude number


## The release still has the bug

In [13]:
!git switch release

Switched to branch 'release'


In [14]:
# This one fails on purpose: the bug the customer reported.
!"{sys.executable}" test_hydrostatics.py

Traceback (most recent call last):
  File "/home/emircem/git-course/07-cherry-pick/test_hydrostatics.py", line 4, in <module>
    assert 0.21 < froude_number(6.17, 80.0) < 0.23
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError


## Picking the fix

Find the fix in the log of `main`:

In [15]:
!git log --oneline main

539c065 (main) Add a rough wetted surface estimate
257f990 Add tonnes per centimetre immersion
91efc42 Fix the Froude number: take the square root
4d94b4f (HEAD -> release) Add volume and the Froude number


Normally you copy its id (the short hash) from there. The id is different
on every computer, so in this notebook we name it `main~2` instead: "two
commits before the end of `main`".

In [16]:
!git cherry-pick main~2

[release 3f897fc] Fix the Froude number: take the square root
 Date: Mon Oct 19 23:43:29 2026 +0200
 1 file changed, 1 insertion(+), 1 deletion(-)


In [17]:
!"{sys.executable}" test_hydrostatics.py

all checks passed


In [18]:
!git log --oneline --graph --all

* 3f897fc (HEAD -> release) Fix the Froude number: take the square root
| * 539c065 (main) Add a rough wetted surface estimate
| * 257f990 Add tonnes per centimetre immersion
| * 91efc42 Fix the Froude number: take the square root
|/  
* 4d94b4f Add volume and the Froude number


The release is fixed, and it did **not** get the two new features. The fix
now exists twice, as two commits with different ids: cherry-pick copies.

## A conflict

Cherry-picking can conflict like a merge. Let's try to pick the wetted
surface feature, which was written after `tonnes_per_cm`:

In [19]:
# This one fails on purpose: a conflict.
!git cherry-pick main

Auto-merging hydrostatics.py
CONFLICT (content): Merge conflict in hydrostatics.py
error: could not apply 539c065... Add a rough wetted surface estimate
hint: After resolving the conflicts, mark them with
hint: "git add/rm <pathspec>", then run
hint: "git cherry-pick --continue".
hint: You can instead skip this commit with "git cherry-pick --skip".
hint: To abort and get back to the state before "git cherry-pick",
hint: run "git cherry-pick --abort".
hint: Disable this message with "git config set advice.mergeConflict false"


We don't want that feature in the release anyway, so we give up:

In [20]:
!git cherry-pick --abort
!git status

On branch release
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	__pycache__/

nothing added to commit but untracked files present (use "git add" to track)


## When to use it

- **Yes:** copying a bug fix to a release branch.
- **Yes:** moving a commit you made on the wrong branch.
- **No:** instead of merging a whole branch. Use `git merge` for that.

## Summary

| I want to… | Command |
| - | - |
| copy one commit here | `git cherry-pick <commit>` |
| give up after a conflict | `git cherry-pick --abort` |
| continue after fixing a conflict | `git cherry-pick --continue` |

## Exercises

1. Make a commit on the wrong branch, then copy it to the right one with
   cherry-pick.
2. Mark the fixed release with a tag: `git tag v1.0.1`, then
   `git log --oneline`.

Next: [8. Submodules](08-submodules.ipynb)